<a href="https://colab.research.google.com/github/leminhohoho/context-bert4rec/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install torch-geometric sentence-transformers

## 1. Model

In [3]:
import torch.nn as nn
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [5]:
dev = True

In [6]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()

        self.pe = nn.Embedding(max_len, d_model) # T,C

    def forward(self, x):
        batch_size = x.size(0)
        return self.pe.weight.unsqueeze(0).repeat(batch_size, 1, 1) # T,C -> B,T,C

class BERT4RecEmbedding(nn.Module):
    def __init__(self, embed_size, max_len, dropout=0.1):
        super().__init__()

        self.pe = PositionalEmbedding(max_len, embed_size)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        pos = self.pe(x) # B,T
        return self.dropout(x + pos) # B,T

if __name__ == "__main__" and dev:
    max_len = 5
    d_model = 4
    batch_size = 2

    model = BERT4RecEmbedding(d_model,max_len)
    x = torch.randn(batch_size, max_len, d_model)

    out = model(x)

    print(x.shape)
    print(x)
    print(out.shape)
    print(out)

    # NOTE: Padding example
    x = torch.randn(batch_size, max_len-2, d_model)
    x = torch.nn.functional.pad(x, (0 , 0, 0, 2))
    padding_mask = (x.abs().sum(dim=-1) == 0)

    print(x.shape)
    print(x)
    print(padding_mask.shape)
    print(padding_mask)


torch.Size([2, 5, 4])
tensor([[[ 1.6693, -0.5047, -1.8117, -0.3503],
         [ 0.7680,  1.2056,  0.3227, -1.2426],
         [ 0.8119, -1.8126,  0.2656, -0.8294],
         [ 0.5566, -0.5532,  0.2169, -0.6309],
         [-1.1456, -0.2377, -0.3980,  0.7498]],

        [[ 1.7180,  2.5945, -0.1167,  0.3450],
         [-0.0329,  1.1167,  0.4203,  1.8825],
         [-0.1690, -0.3096, -0.8600,  1.0990],
         [ 0.1423, -0.6774,  0.3373, -2.4740],
         [-0.7073, -1.0507, -1.3130, -0.1640]]])
torch.Size([2, 5, 4])
tensor([[[ 2.4612,  2.3383, -2.2613, -0.6774],
         [ 1.3566, -0.0641, -0.6372, -0.7326],
         [ 1.4973, -2.6114, -1.9063,  0.8593],
         [ 0.0000,  0.5305,  0.6345,  1.1092],
         [-2.2546,  0.0000, -0.1244,  0.0184]],

        [[ 2.5154,  5.7819, -0.3779,  0.0953],
         [ 0.4667, -0.1628, -0.0000,  2.7398],
         [ 0.4075, -0.9414, -3.1571,  3.0020],
         [-0.3261,  0.3925,  0.7683, -0.9388],
         [-1.7676, -0.2583, -1.1411, -0.0000]]], grad_fn=

In [7]:
class BERT(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, dropout=0.1):
        super().__init__()
        self.embedding = BERT4RecEmbedding(
            embed_size=hidden,
            max_len=max_len,
            dropout=dropout,
        )

        self.mask_token = nn.Parameter(torch.randn(hidden))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=n_heads,
            batch_first=True,
            dropout=dropout,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=n_layers,
        )

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, padding_mask=None, masked_positions=None):
        if padding_mask is None:
            padding_mask = (x.abs().sum(dim=-1) == 0)

        if masked_positions is not None: # B,T
            x = torch.where(
                masked_positions.unsqueeze(-1), # B,T,1
                self.mask_token.view(1, 1, -1), # 1,1,C
                x,
            )

        x = self.embedding(x)

        x = self.encoder(x, src_key_padding_mask=padding_mask)
        x = self.dropout(x)

        return x

if __name__ == "__main__" and dev:
    batch_size = 2
    max_len = 4
    hidden = 32
    n_layers = 3
    n_heads = 4

    model = BERT(max_len=max_len, hidden=hidden, n_layers=n_layers, n_heads=n_heads)

    x = torch.randn(batch_size, max_len, hidden)

    out = model(x)

    print(out.shape)
    print(out)

torch.Size([2, 4, 32])
tensor([[[-1.1833, -0.7409,  1.8747,  0.1235,  0.2901,  0.0000,  0.6725,
          -0.1067,  1.1720, -2.0470, -0.0000, -0.1013, -0.3877,  0.1148,
          -1.6886,  1.1482, -0.1662,  0.0519, -0.3831, -0.3054,  1.0982,
           0.0722, -1.7760,  0.2482,  0.4906,  0.3684,  0.0113, -0.0400,
          -1.3193, -0.0000,  0.2033,  2.1120],
         [ 0.8421,  0.1750,  1.0520, -0.6919, -0.2308,  0.0000, -0.7306,
          -0.1409,  2.2645, -2.0298, -0.0000, -0.5597, -1.9232,  1.2576,
           0.6298,  1.1080, -0.7148, -0.0000, -0.0000,  1.4187, -0.5708,
           0.6362, -0.1057,  1.0741,  0.1954,  0.6542, -0.6131, -0.5175,
          -1.5233,  0.4391, -0.9570,  0.0000],
         [ 0.6705, -0.9012, -0.9009,  1.4730, -0.0134,  1.6210, -0.5058,
          -0.5717,  0.0000,  0.0802, -0.5156,  0.2757, -0.6595, -1.1830,
          -1.3573,  1.0530, -0.3421,  0.1211, -1.6069,  0.5166, -0.8328,
          -1.2468,  1.7059,  2.0122, -1.8764,  0.3389,  2.1865, -1.3514,
       

In [8]:
import torch.nn.functional as F

class UserItemEncoder(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, dropout=0.1):
        super().__init__()

        self.encoder = BERT(
            max_len=max_len,
            n_layers=n_layers,
            n_heads=n_heads,
            hidden=hidden,
            dropout=dropout,
        )

        self.dropout = nn.Dropout(p=dropout)

        self.proj = nn.Linear(hidden, hidden)

    def forward(self, x):
        x = self.encoder(x)
        h = self.proj(x[:, -1, :])

        return h

class UserInteractionEncoder(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, embed_size, dropout=0.1):
        super().__init__()

        self.encoder = BERT(
            max_len=max_len,
            n_layers=n_layers,
            n_heads=n_heads,
            hidden=hidden,
            dropout=dropout,
        )

        self.dropout = nn.Dropout(p=dropout)

        self.proj = nn.Sequential(
            nn.Linear(hidden, embed_size*2),
            nn.ReLU(),
            self.dropout,
            nn.Linear(embed_size*2, embed_size*2),
            nn.ReLU(),
            self.dropout,
            nn.Linear(embed_size*2, embed_size),
        )

    def forward(self, x):
        x = self.encoder(x)
        h = self.proj(x[:, -1, :])

        return h


## 2. Training

In [9]:

movies_df = pd.read_csv("/content/drive/MyDrive/datasets/ml-20m/movies.csv")
ratings_df = pd.read_csv("/content/drive/MyDrive/datasets/ml-20m/ratings.csv")

print(movies_df.shape)
print(movies_df.head().to_string())
print(ratings_df.shape)
print(ratings_df.head().to_string())

(27278, 3)
   movieId                               title                                       genres
0        1                    Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy
1        2                      Jumanji (1995)                   Adventure|Children|Fantasy
2        3             Grumpier Old Men (1995)                               Comedy|Romance
3        4            Waiting to Exhale (1995)                         Comedy|Drama|Romance
4        5  Father of the Bride Part II (1995)                                       Comedy
(20000263, 4)
   userId  movieId  rating   timestamp
0       1        2     3.5  1112486027
1       1       29     3.5  1112484676
2       1       32     3.5  1112484819
3       1       47     3.5  1112484727
4       1       50     3.5  1112484580


In [10]:
sentence_encoder = SentenceTransformer("all-MiniLM-L6-v2")

movies_df["combined_text"] = (
    movies_df["title"].fillna("") + " - " +
    movies_df["genres"].fillna("")
)

movies_df["embedding"] = list(
    sentence_encoder.encode(
        movies_df["combined_text"].tolist(),
        batch_size=256,
        show_progress_bar=True,
        device=device,
    )
)

movies_df[movies_df["movieId"] == 31696]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

,movieId,title,genres,combined_text,embedding
9757,31696,Constantine (2005),Action|Fantasy|Horror|Thriller,Constantine (2005) - Action|Fantasy|Horror|Thr...,"[-0.0649995, -0.08383038, -0.034668762, 0.0378..."


In [11]:
class CloveTaskItemEncoderDataset(Dataset):
    def __init__(self, movies_df, ratings_df, max_len, mask_rate):
        self.movies_df = movies_df.set_index("movieId")
        self.ratings_df = ratings_df
        self.max_len = max_len
        self.mask_rate = mask_rate
        self.seqs = []

        users_ratings = [
            user_df for _, user_df in (
                self.ratings_df
                    .sort_values("timestamp")
                    .groupby("userId", sort=False)
                )
        ]

        for user_ratings in users_ratings:
            self.seqs.extend([user_ratings[i:i+max_len]["movieId"].to_list() for i in range(len(user_ratings) - max_len + 1)])

    def __len__(self):
        return len(self.seqs)


    def __getitem__(self, idx):
        seq = self.seqs[idx]

        mask = (torch.randperm(max_len) - int(max_len * (self.mask_rate)) >= 0).float()

        return torch.from_numpy(np.stack(self.movies_df.loc[seq]["embedding"].to_list())), mask

In [12]:
lr=1e-4
batch_size= 512
epochs=10
max_len=20
hidden=384
n_layers=16
n_heads=8
dropout=0.1
mask_rate=0.2
val_iter=10

In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

In [14]:
test_ratings_df = ratings_df.sort_values("timestamp")[:1000000]

split_idx = int(len(test_ratings_df) * 0.8)

# NOTE: Limit rating for testing
train_ratings_df = test_ratings_df.iloc[:split_idx]
val_ratings_df = test_ratings_df.iloc[split_idx:]

print(train_ratings_df)
print(val_ratings_df)

          userId  movieId  rating  timestamp
4182421    28507     1176     4.0  789652004
18950930  131160       21     3.0  789652009
18950936  131160       47     5.0  789652009
18950979  131160     1079     3.0  789652009
7754002    53434       19     1.0  822873600
...          ...      ...     ...        ...
17153864  118651       11     4.0  839941769
18268307  126363      480     4.0  839941770
18268297  126363      356     2.0  839941770
17776244  122901      586     3.0  839941773
17776245  122901      587     3.0  839941773

[800000 rows x 4 columns]
          userId  movieId  rating  timestamp
17776252  122901      597     3.0  839941773
16897485  116888      728     3.0  839941790
16897387  116888       52     3.0  839941790
16897466  116888      531     4.0  839941807
17776243  122901      539     3.0  839941817
...          ...      ...     ...        ...
1372848     9294      592     3.0  842369924
1372805     9294      296     5.0  842369924
1372818     9294      380   

In [15]:
train_ds = CloveTaskItemEncoderDataset(movies_df, train_ratings_df, max_len, mask_rate)
val_ds = CloveTaskItemEncoderDataset(movies_df, val_ratings_df, max_len, mask_rate)

In [16]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
)

val_loader = DataLoader(
    dataset=val_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
)

In [17]:
user_items_bert = BERT(
    max_len=max_len,
    n_layers=n_layers,
    n_heads=n_heads,
    hidden=hidden,
    dropout=dropout,
)

user_items_bert.to(device)

print(f"Number of parameters: {sum(p.numel() for p in user_items_bert.parameters() if p.requires_grad)}")

Number of parameters: 34699136


In [18]:
train_losses = []
val_losses = []

for epoch in range(epochs):
    user_items_bert.train()

    for batch_idx, (items_seq, mask) in enumerate(train_loader):
        items_seq, mask = next(iter(train_loader))
        items_seq = items_seq.to(device)
        mask = mask.to(device)

        out = user_items_bert(items_seq, mask)

        pred = out[mask.bool()]
        target = items_seq[mask.bool()]

        loss = criterion(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

        if batch_idx % 100 == 0:
            print(f"[Train] Epoch: {epoch} - Batch: {batch_idx}, Loss: {loss.item()}, Average loss: {sum(train_losses)/len(train_losses)}")

    if epoch % val_iter == 0 and epoch != 0:
        user_items_bert.eval()

        with torch.no_grad():
            items_seq, mask = next(iter(val_loader))
            items_seq = items_seq.to(device)
            mask = mask.to(device)

            out = user_items_bert(items_seq, mask)

            pred = out[mask.bool()]
            target = items_seq[mask.bool()]

            loss = criterion(pred, target)

            val_losses.append(loss.item())

            print(f"[Validation] Epoch: {epoch}, Loss: {loss.item()}, Average loss: {sum(val_losses)/len(val_losses)}")

[Train] Epoch: 0 - Batch: 0, Loss: 1.1103012561798096, Average loss: 1.1103012561798096
[Train] Epoch: 0 - Batch: 100, Loss: 1.1105173826217651, Average loss: 1.11034097057758
[Train] Epoch: 0 - Batch: 200, Loss: 1.1103339195251465, Average loss: 1.1103366346501593
[Train] Epoch: 0 - Batch: 300, Loss: 1.1104878187179565, Average loss: 1.1103537656936138
[Train] Epoch: 0 - Batch: 400, Loss: 1.1096709966659546, Average loss: 1.1103441307966846
[Train] Epoch: 0 - Batch: 500, Loss: 1.1106394529342651, Average loss: 1.1103419848306926
[Train] Epoch: 0 - Batch: 600, Loss: 1.1104320287704468, Average loss: 1.1103415372169354
[Train] Epoch: 0 - Batch: 700, Loss: 1.110122561454773, Average loss: 1.1103416897260854
[Train] Epoch: 0 - Batch: 800, Loss: 1.109764814376831, Average loss: 1.110335299435924
[Train] Epoch: 0 - Batch: 900, Loss: 1.1102925539016724, Average loss: 1.1103332169709539
[Train] Epoch: 0 - Batch: 1000, Loss: 1.1099927425384521, Average loss: 1.1103360144408432
[Train] Epoch: 1

KeyboardInterrupt: 